# Section 1 – Collect the data
**STUDENT notebook: complete the TODO cells**

| | |
|---|---|
| Connects to | Topic 1 – HTTP & REST |
| Estimated time | 50 minutes |
| Produces | `data/pokemon_raw.csv` |

**Goal:** Call a public game API and read its JSON, then build the full ~1000-character table from the same project's published CSV files.

### How to work in this notebook
- 🟦 **Worked sample** cells run as they are. Read them, run them, change them.
- 🟨 **Your turn** cells contain `# TODO` and `...`. Replace `...` with your code.
- ✅ **Check** cells print `PASS` / `FAIL`. Do not move on while you see a `FAIL`.
- Stuck? Open the same file in the `solution/` folder, but try first.

## Step 0 – Setup

In [ ]:
import importlib.util
import subprocess
import sys

# 1) Make sure the libraries exist in THIS kernel (installs only what is missing)
REQUIRED = {"requests": "requests", "pandas": "pandas", "numpy": "numpy",
            "scikit-learn": "sklearn", "matplotlib": "matplotlib", "pillow": "PIL",
            "joblib": "joblib", "fastapi": "fastapi", "uvicorn": "uvicorn",
            "httpx": "httpx", "pydantic": "pydantic"}
missing = [pkg for pkg, mod in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing, "-> restart the kernel if an import below fails")
else:
    print("All dependencies are available")

# 2) Corporate networks with HTTPS inspection: use the Windows certificate store if available
try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore active (Windows certificate store)")
except ImportError:
    pass

# 3) Lab folders (this notebook lives in student/ or solution/, so the lab root is one level up)
from pathlib import Path
import json

ROOT = Path.cwd().parent
DATA, MODELS, OUT = ROOT / "data", ROOT / "models", ROOT / "outputs"
for folder in (DATA, MODELS, OUT):
    folder.mkdir(exist_ok=True)


def check(name, ok, detail=""):
    """Self-check helper: prints PASS / FAIL."""
    print(("PASS" if ok else "FAIL"), "-", name, ("-> " + str(detail)) if detail else "")


print("Lab root:", ROOT)

## What we are building
The whole lab builds a **Character Card Studio**: a service that returns a collectible card (PNG) for a game character,
with two machine-learning predictions printed on it.

```
PokéAPI ──requests──▶ CSV ──▶ clean ──▶ 2 models ──▶ FastAPI ──▶ PNG card
 (this section)
```

In this section we only do the first arrow: **collect data**.

| Source | What we use | Why |
|---|---|---|
| PokéAPI `/pokemon/{id}` and `/pokemon-species/{id}` | JSON, one character per request | **practice** REST, JSON, errors, threads (12 characters) |
| PokéAPI public CSV files on GitHub | 5 small tables, one download each | **the right tool** to get all ~1000 characters quickly |

Both are free and need no key. A key lesson of this section: *an API is not always the best way to get bulk data.*
One `/pokemon/{id}` answer is ~280 KB and takes ~3 s, so 1025 of them would be ~270 MB and more than 10 minutes.

## Step 1 – Are the services reachable?
If you are on a corporate network and see an **SSL error**, the network is re-signing HTTPS traffic.
The fix is to let Python use the Windows certificate store:

```
pip install truststore
```
then restart the kernel and run Step 0 again (the setup cell activates it automatically).

In [ ]:
import requests

for label, url in [("PokéAPI", "https://pokeapi.co/api/v2/pokemon/1"),
                   ("GitHub raw files", "https://raw.githubusercontent.com/PokeAPI/pokeapi/master/data/v2/csv/stats.csv")]:
    try:
        r = requests.get(url, timeout=15)
        print(f"{label:<17} reachable, status: {r.status_code}")
    except requests.exceptions.SSLError:
        print(f"{label:<17} SSL error: run  pip install truststore  then restart the kernel and re-run Step 0")
    except requests.exceptions.RequestException as e:
        print(f"{label:<17} network problem:", e)

## Step 2 – One GET request, step by step
Recall Topic 1: a request has a **method + URL**; the response has a **status code**, **headers** and a **body** (JSON).

🟦 **Worked sample** (a different API, D&D monsters):

In [ ]:
r = requests.get("https://www.dnd5eapi.co/api/2014/monsters/goblin", timeout=10)
print("status      :", r.status_code)
print("content-type:", r.headers["content-type"])

monster = r.json()                       # JSON text -> Python dict
print("name        :", monster["name"])
print("hit points  :", monster["hit_points"])
print("first keys  :", list(monster)[:8])

🟨 **Your turn.** Do the same for Pikachu (id `25`) on PokéAPI: `https://pokeapi.co/api/v2/pokemon/25`.

In [ ]:
# TODO: Send a GET request to the Pikachu URL with a 10 second timeout. Keep the response in `resp`.
# HINT: requests.get(url, timeout=10)
...  # <- write your code here

# TODO: Convert the body to a Python dict called `pikachu_raw`
# HINT: resp.json()
...  # <- write your code here

check("status code is 200", resp.status_code == 200)
check("name is pikachu", pikachu_raw["name"] == "pikachu")

## Step 3 – Read the JSON structure
The body is **nested**: dicts inside lists inside dicts. Before writing any code, look at it.

In [ ]:
print("Top-level keys of the goblin:", list(monster.keys())[:10], "...")
# a list of dicts: each action has a name
print([a["name"] for a in monster["actions"]])
# a list of dicts, each with a nested dict
print(monster["armor_class"])

In [ ]:
print("Pikachu top-level keys:", list(pikachu_raw.keys())[:12], "...")
print()
print("stats (list of dicts), first item:")
print(json.dumps(pikachu_raw["stats"][0], indent=2))

🟨 **Your turn.** From `pikachu_raw` extract:
1. a dict `stats` like `{"hp": 35, "attack": 55, ...}` (hint: loop over `pikachu_raw["stats"]`, name is `s["stat"]["name"]`, value is `s["base_stat"]`)
2. a list `types` with the type names (hint: `pikachu_raw["types"]`)
3. the official artwork URL `artwork` (hint: `sprites` → `other` → `official-artwork` → `front_default`)

In [ ]:
# TODO: Build the dict `stats` with a dict comprehension
# HINT: {s["stat"]["name"]: s["base_stat"] for s in ...}
...  # <- write your code here

# TODO: Build the list `types`
...  # <- write your code here

# TODO: Get the artwork URL
...  # <- write your code here

print(stats)
print(types)
print(artwork)
check("6 stats found", len(stats) == 6)
check("speed is 90", stats.get("speed") == 90)
check("artwork is a URL", str(artwork).startswith("http"))

## Step 4 – A safe GET helper (timeouts, retries, 404)
Real networks fail. A good client **never trusts a single request**. Pattern: try a few times, wait a little longer each
time (*backoff*), and treat `404` as "does not exist" instead of crashing.

🟦 **Worked sample** (read it, we reuse it in the next steps):

In [ ]:
import time


def get_json(url, retries=3):
    """GET with timeout, retries and backoff. Returns a dict, or None if it does not exist / keeps failing."""
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=20)
            if r.status_code == 404:
                return None                       # the resource does not exist: no point in retrying
            r.raise_for_status()                  # 5xx -> exception -> retry
            return r.json()
        except requests.exceptions.RequestException:
            time.sleep(1.5 * (attempt + 1))       # wait 1.5 s, 3 s, 4.5 s ...
    return None


print(get_json("https://www.dnd5eapi.co/api/2014/monsters/orc")["name"])
print(get_json("https://www.dnd5eapi.co/api/2014/monsters/does-not-exist"))

## Step 5 – Turn one character into one table row
Two endpoints per character, so `parse_pokemon(raw_pokemon, raw_species)` flattens two nested JSON documents into one flat
dict (one future table row).

Generations are text (`"generation-iii"`), models need numbers, so we provide the mapping:

In [ ]:
GEN = {"generation-i": 1, "generation-ii": 2, "generation-iii": 3, "generation-iv": 4, "generation-v": 5,
       "generation-vi": 6, "generation-vii": 7, "generation-viii": 8, "generation-ix": 9}
BASE = "https://pokeapi.co/api/v2"

In [ ]:
def parse_pokemon(raw_p, raw_s):
    """Flatten the two raw JSON documents into one dict (one row)."""
    # TODO: stats as a dict {name: base_stat}
    ...  # <- write your code here
    # TODO: list of type names
    ...  # <- write your code here
    # TODO: the six stats as a dict. Careful: PokéAPI writes "special-attack" with a hyphen, our column is special_attack
    ...  # <- write your code here
    # TODO: second type, or None if the character has only one type
    ...  # <- write your code here
    return {
        "id": raw_p["id"],
        "name": raw_p["name"],
        **stat_fields,
        "height": raw_p["height"],
        "weight": raw_p["weight"],
        "base_experience": raw_p["base_experience"],          # our REGRESSION target
        "type1": types[0],
        "type2": type2,
        "generation": GEN[raw_s["generation"]["name"]],
        "is_legendary": raw_s["is_legendary"],                # our CLASSIFICATION target
        "is_mythical": raw_s["is_mythical"],
        "artwork_url": raw_p["sprites"]["other"]["official-artwork"]["front_default"],
    }


# quick test with Pikachu
row = parse_pokemon(get_json(f"{BASE}/pokemon/25"), get_json(f"{BASE}/pokemon-species/25"))
print(row)
check("row has 17 fields", len(row) == 17, len(row))
check("generation is a number", row["generation"] == 1)

## Step 6 – A small REST collection with threads
Two requests per character (pokemon + species). To save time we use a **thread pool**: a few requests in flight at once.
Do not raise `MAX_WORKERS` above 8: public APIs are shared.

🟦 **Worked sample**: concurrency with 5 D&D monsters.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

names = ["goblin", "orc", "wolf", "zombie", "skeleton"]


def hp_of(name):
    return name, get_json(f"https://www.dnd5eapi.co/api/2014/monsters/{name}")["hit_points"]


with ThreadPoolExecutor(max_workers=3) as pool:
    print(dict(pool.map(hp_of, names)))

🟨 **Your turn.** Write `fetch_character(pid)` (two GETs + parse) and run it for the first 12 characters.

In [ ]:
SAMPLE_IDS = range(1, 13)
MAX_WORKERS = 6


def fetch_character(pid):
    # TODO: download the pokemon and the species documents; return None if any of them is missing
    ...  # <- write your code here
    return parse_pokemon(raw_p, raw_s)


# TODO: run fetch_character for every id in SAMPLE_IDS with a ThreadPoolExecutor and keep the non-None results in `rest_rows`
# HINT: with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool: results = list(pool.map(fetch_character, SAMPLE_IDS))
...  # <- write your code here

import pandas as pd

rest_df = pd.DataFrame(rest_rows)
print(f"{len(rest_df)} characters in {time.time() - started:.0f} s")
rest_df[["id", "name", "hp", "attack", "speed", "base_experience", "type1", "is_legendary"]].head()

In [ ]:
check("12 characters downloaded", len(rest_df) == 12, len(rest_df))
check("first one is bulbasaur", rest_df.sort_values("id").iloc[0]["name"] == "bulbasaur")

## Step 7 – When an API is the wrong tool
We need ~1000 characters. How much would REST cost? Measure it with the Pikachu response from Step 2.

In [ ]:
# TODO: size of ONE pokemon response in KB (resp.content is the raw bytes) and the projected size for 1025 characters in MB
...  # <- write your code here

print(f"one response: {size_kb:.0f} KB  ->  1025 responses: about {projected_mb:.0f} MB (plus the species calls)")
check("a response is larger than 100 KB", size_kb > 100, f"{size_kb:.0f} KB")

**Bulk alternative.** The PokéAPI project also publishes its database as plain **CSV files** (a "bulk export").
One `GET` per table, a few KB each. The price: the data is *normalized* (split in several tables), so we must **join** it ourselves.

| File | One row is... | Columns we need |
|---|---|---|
| `pokemon.csv` | a character (and its alternate forms) | `id, identifier, species_id, height, weight, base_experience, is_default` |
| `pokemon_stats.csv` | one stat of one character (**long** format: 6 rows per character) | `pokemon_id, stat_id, base_stat` |
| `pokemon_species.csv` | a species | `id, generation_id, is_legendary, is_mythical` |
| `pokemon_types.csv` | one type of one character | `pokemon_id, type_id, slot` |
| `types.csv` | a type name | `id, identifier` |

🟦 **Worked sample**: `pandas.read_csv` can read straight from a URL (it performs the GET for us).

In [ ]:
CSV_BASE = "https://raw.githubusercontent.com/PokeAPI/pokeapi/master/data/v2/csv/"

stat_names = pd.read_csv(CSV_BASE + "stats.csv")
print(stat_names[["id", "identifier"]].head(8).to_string(index=False))

## Step 8 – Load the tables and reshape the stats
The stats table is **long**: `(pokemon_id=25, stat_id=1, base_stat=35)`, `(25, 2, 55)`, ... We need it **wide**: one row per
character, one column per stat. That reshape is called a **pivot**.

🟦 **Worked sample** (toy table):

In [ ]:
long_toy = pd.DataFrame({"hero": ["A", "A", "B", "B"], "stat": ["hp", "speed", "hp", "speed"], "value": [80, 50, 120, 20]})
print(long_toy.pivot(index="hero", columns="stat", values="value"))

In [ ]:
# TODO: load the five tables from CSV_BASE: pokemon, pokemon_stats, pokemon_species, pokemon_types and types
# HINT: pd.read_csv(CSV_BASE + "pokemon.csv")
...  # <- write your code here

STAT_NAMES = {1: "hp", 2: "attack", 3: "defense", 4: "special_attack", 5: "special_defense", 6: "speed"}

# TODO: keep only stat ids 1-6, pivot to one row per character (columns = stat names), and rename pokemon_id to id
# HINT: stats_long[stats_long["stat_id"].isin(STAT_NAMES)].pivot(index="pokemon_id", columns="stat_id", values="base_stat")
...  # <- write your code here

print(wide.shape)
wide.head(3)

In [ ]:
check("six stat columns + id", list(wide.columns[1:]) == list(STAT_NAMES.values()), list(wide.columns))
check("pikachu speed is 90", int(wide.loc[wide["id"] == 25, "speed"].iloc[0]) == 90)

## Step 9 – Join everything into one table
Join rule: **one row per default character** (`is_default == 1`; the other rows are alternate forms such as "mega" or "gmax").
Types need two small steps: `type_id` → name, then slot 1 → `type1`, slot 2 → `type2` (a character may have only one type).

In [ ]:
ART = "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/"
COLUMNS = ["id", "name", "hp", "attack", "defense", "special_attack", "special_defense", "speed", "height", "weight",
           "base_experience", "type1", "type2", "generation", "is_legendary", "is_mythical", "artwork_url"]

# TODO: Series `type1` and `type2` indexed by pokemon_id: map type_id to the type name, then split by slot (1 and 2)
# HINT: ptypes["type"] = ptypes["type_id"].map(types.set_index("id")["identifier"])
...  # <- write your code here

# TODO: default characters only; join the wide stats (on id), type1 (inner), type2 (LEFT: some have no second type) and the species columns
# HINT: species has its own `id`; join pokemon.species_id with species.id
...  # <- write your code here

df["artwork_url"] = ART + df["id"].astype(str) + ".png"
df = df[COLUMNS].sort_values("id").reset_index(drop=True)
print(df.shape)
df.head()

## Step 10 – Save, and cross-check the two sources
Two independent sources (REST JSON and bulk CSV) should agree. If they do, we trust the joins.

In [ ]:
CSV_PATH = DATA / "pokemon_raw.csv"
# TODO: save the table without the index column
# HINT: df.to_csv(path, index=False)
...  # <- write your code here

# compare the 12 characters collected through REST with the same rows of the bulk table
bulk = df.set_index("id").loc[rest_df["id"]]
rest = rest_df.set_index("id")
for col in ["hp", "attack", "defense", "special_attack", "special_defense", "speed", "height", "weight",
            "base_experience", "generation"]:
    check(f"REST and bulk agree on {col}", (bulk[col].values == rest[col].values).all())
check("REST and bulk agree on is_legendary", (bulk["is_legendary"].astype(int).values == rest["is_legendary"].astype(int).values).all())
check("REST and bulk agree on type1", (bulk["type1"].values == rest["type1"].values).all())
check("same artwork URL", (bulk["artwork_url"].values == rest["artwork_url"].values).all())

In [ ]:
saved = pd.read_csv(CSV_PATH)
expected = {"id", "name", "hp", "attack", "defense", "special_attack", "special_defense", "speed", "height",
            "weight", "base_experience", "type1", "type2", "generation", "is_legendary", "is_mythical", "artwork_url"}
check("all 17 columns exist", expected == set(saved.columns), set(saved.columns) ^ expected or "")
check("ids are unique", saved["id"].is_unique)
check("about 1000 characters", len(saved) >= 1000, len(saved))
print("Missing values per column (only the non-zero ones):")
print(saved.isna().sum()[saved.isna().sum() > 0])

## Questions to think about (write the answers in a cell below)
1. Why is a 404 handled differently from a 500 in `get_json`?
2. REST gave us 12 characters in seconds; the CSV gave us 1025. When would you still prefer the REST API? (Think: freshness, one record, size.)
3. Why did we use a **left** join for `type2` and an inner join for `type1`?
4. Which column will be our **regression** target and which one the **classification** target? Which column has missing values, and why?

**Next:** Section 2 – explore and clean the table.